# Qdrant Ingestion Notebook

This notebook walks through ingesting the mock video captions into Qdrant and
validating that **all** videos and snapshots from `data/mock_video_captions.json`
are present in the collection after ingestion.

In [6]:
import os
os.chdir("/Users/vasim/Programming/ai-engineering/video-rag-agent")

In [7]:
from video_rag import (
    load_mock_data,
    count_points,
    search_by_video_id,
)
from video_rag.ingest import get_client
from video_rag.config import settings

# Reload mock data from disk to know what *should* be in Qdrant.
data = load_mock_data()
expected_videos = data["videos"]
expected_video_ids = [v["video_id"] for v in expected_videos]
expected_snapshot_counts = {v["video_id"]: len(v["snapshots"]) for v in expected_videos}
expected_total_snapshots = sum(expected_snapshot_counts.values())

print(f"Collection: {settings.collection_name}")
print(f"Qdrant URL:  {settings.qdrant_url}")
print(f"Expected videos:    {len(expected_video_ids)}")
print(f"Expected snapshots: {expected_total_snapshots}")


Collection: video_captions
Qdrant URL:  http://localhost:6333
Expected videos:    17
Expected snapshots: 100


## Ingest the mock data into Qdrant

Skips ingestion if the collection already contains the expected number of points
so the notebook is idempotent.

In [8]:
from video_rag import recreate_collection, ingest_mock_data, count_points

client = get_client()
existing = count_points(client=client)
print(f"Points currently in collection: {existing}")

if existing != expected_total_snapshots:
    print("Collection size mismatch — recreating and re-ingesting...")
    recreate_collection(client=client)
    n = ingest_mock_data(client=client)
    print(f"Ingested {n} points")
else:
    print("Collection already has the expected number of points — skipping ingest.")

Points currently in collection: 100
Collection already has the expected number of points — skipping ingest.


## Validate that all videos were ingested

For every video in the mock dataset we:
1. Scroll all its snapshots out of Qdrant via `search_by_video_id`.
2. Compare the count against the source data.
3. Cross-check `video_title`, `category`, and the set of `end_seconds`
   values (which equal the source `timestamp_seconds`) so a duplicated
   point for another video can never silently "satisfy" the check.


In [9]:
def fetch_video_payloads(video_id: str) -> list[dict]:
    """Return a list of payload dicts for every snapshot of `video_id`."""
    return [p.payload for p in search_by_video_id(video_id, client=client)]


def expected_ends(video: dict) -> set[int]:
    """Map source `timestamp_seconds` to the `end_seconds` Qdrant stores."""
    return {s["timestamp_seconds"] for s in video["snapshots"]}


rows = []
all_ok = True
for video in expected_videos:
    payloads = fetch_video_payloads(video["video_id"])
    got_n = len(payloads)
    want_n = expected_snapshot_counts[video["video_id"]]

    # Cross-check title/category on the first payload (every snapshot shares them).
    title_ok = bool(payloads) and payloads[0]["video_title"] == video["title"]
    category_ok = bool(payloads) and payloads[0]["category"] == video["category"]
    # The ingestion code stores the source `timestamp_seconds` as `end_seconds`.
    ends_ok = {p["end_seconds"] for p in payloads} == expected_ends(video)

    ok = (got_n == want_n) and title_ok and category_ok and ends_ok
    all_ok = all_ok and ok
    rows.append(
        {
            "video_id": video["video_id"],
            "category": video["category"],
            "expected": want_n,
            "found": got_n,
            "title_ok": "✅" if title_ok else "❌",
            "category_ok": "✅" if category_ok else "❌",
            "ends_ok": "✅" if ends_ok else "❌",
            "status": "✅" if ok else "❌",
        }
    )

# Render as a markdown table
header = (
    f"| {'video_id':<12} | {'category':<12} | {'expected':>8} | "
    f"{'found':>5} | title | category | ends | status |\n"
    "|--------------|--------------|----------|-------|-------|----------|-------|--------|"
)
body = "\n".join(
    f"| {r['video_id']:<12} | {r['category']:<12} | {r['expected']:>8} | "
    f"{r['found']:>5} |   {r['title_ok']}   |    {r['category_ok']}     |  {r['ends_ok']}  |   {r['status']}   |"
    for r in rows
)
print(header + "\n" + body)
print()
print("ALL VIDEOS OK ✅" if all_ok else "VALIDATION FAILED ❌")

| video_id     | category     | expected | found | title | category | ends | status |
|--------------|--------------|----------|-------|-------|----------|-------|--------|
| video_001    | nature       |        7 |     7 |   ✅   |    ✅     |  ✅  |   ✅   |
| video_002    | food         |        6 |     6 |   ✅   |    ✅     |  ✅  |   ✅   |
| video_003    | technology   |        7 |     7 |   ✅   |    ✅     |  ✅  |   ✅   |
| video_004    | nature       |        5 |     5 |   ✅   |    ✅     |  ✅  |   ✅   |
| video_005    | food         |        6 |     6 |   ✅   |    ✅     |  ✅  |   ✅   |
| video_006    | technology   |        7 |     7 |   ✅   |    ✅     |  ✅  |   ✅   |
| video_007    | urban        |        5 |     5 |   ✅   |    ✅     |  ✅  |   ✅   |
| video_008    | fitness      |        6 |     6 |   ✅   |    ✅     |  ✅  |   ✅   |
| video_009    | music        |        7 |     7 |   ✅   |    ✅     |  ✅  |   ✅   |
| video_010    | animals      |        4 |     4 |   ✅   |    ✅     |  

## Final sanity check

Compare the collection point count returned by Qdrant against the expected
total snapshot count from the mock dataset.

In [10]:
final_count = count_points(client=client)
print(f"Qdrant points:   {final_count}")
print(f"Expected points: {expected_total_snapshots}")
print("Total OK ✅" if final_count == expected_total_snapshots else "Total MISMATCH ❌")

Qdrant points:   100
Expected points: 100
Total OK ✅
